### How to evaluate prompt / prompt Engineering

In [13]:
from anthropic import Anthropic;
import json;


def add_message(messageList,message,role):
    msg={
        "role": role,
        "content":message
    };
    messageList.append(msg);
    

def generateDataset():
    prompt="""
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": json python or regex,
            "solution_criteria" : "key crtiteria for solution"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """
    messageList=[]
    add_message(messageList,prompt,"user")
    add_message(messageList,"```json","assistant")
    answer=chat(messageList,"```") 
    return json.loads(answer);

def chat(messageList,stopSequence=None):
    anthropicClient=Anthropic();
    model='claude-haiku-4-5';
    stop_Sequence=[];
    if stopSequence:
        stop_Sequence.append(stopSequence);
    answer=anthropicClient.messages.create(
        model=model,
        max_tokens=1000,
        messages=messageList,
        stop_sequences=stop_Sequence
    )
    print(answer)
    return answer.content[0].text




In [14]:
from dotenv import load_dotenv;
load_dotenv();

In [ ]:
dateset=generateDataset()

In [16]:
dateset

[{'task': 'Parse an AWS CloudFormation template and extract all S3 bucket logical IDs',
  'format': 'python',
  'solution_criteria': "Function should accept a CloudFormation JSON string and return a list of logical IDs for resources with type 'AWS::S3::Bucket'"},
 {'task': 'Validate that an AWS IAM policy JSON has required statement fields',
  'format': 'json',
  'solution_criteria': 'JSON schema should enforce that each statement contains Version, Effect, Action, and Resource fields; validation should pass for well-formed policies and fail for incomplete ones'},
 {'task': 'Extract AWS EC2 instance IDs from CloudWatch logs using pattern matching',
  'format': 'regex',
  'solution_criteria': "Regular expression should match and capture EC2 instance IDs in format 'i-' followed by 8 or 17 hexadecimal characters from log text"}]

In [17]:
with open("dataset.json","w") as file:
    json.dump(dateset,file,indent=2)

In [18]:
def runPrompt(test_case):
    prompt=f"""
    PLease solve the following task
    {test_case["task"]}

    * Respond only with Python , json or a plain regex
    * Do not add any comments or comemtary or explaination
"""
    messageList=[];
    add_message(messageList,prompt,"user")
    add_message(messageList,"```code","assistant")
    answer=chat(messageList,"```")
    return answer;


In [19]:
def modelGrader(test_case,output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Solution Criteria:
    <Solution Criteria>
    {test_case["solution_criteria"]}
    </Solution Criteria>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
        """
    messageList=[];
    add_message(messageList,eval_prompt,"user");
    add_message(messageList,"```json","assistant");
    eval_output=chat(messageList,"```")
    return json.loads(eval_output);

In [20]:
import re ;
import ast;

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0;

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


In [21]:
def runTestCase(test_case):
    output=runPrompt(test_case)
    codeGrade=-1;
    if test_case["format"]=='json':
        codeGrade=validate_json(output)
    if test_case["format"]=='python':
        codeGrade=validate_python(output)
    else:
        codeGrade=validate_regex(output)

    eval_output=modelGrader(test_case,output)
    score=eval_output["score"];
    reasoning=eval_output["reasoning"]
    return{
        "output":output,
        "test_case" :test_case,
        "score":score,
        "reasoning" : reasoning,
        "Code Grade": codeGrade
    }

In [22]:
def run_eval(dataset):
    results=[]
    for testcase in dataset:
        result=runTestCase(testcase);
        results.append(result);
    return results;

In [ ]:
with open("dataset.json","r") as f:
    dataset=json.load(f);

results=run_eval(dataset);

In [24]:
for result in results:
    print(f"testcase : {result["test_case"]}")
    print(f"output : {result["output"]}")
    print(f"score : {result["score"]}")
    print(f"reasoning : {result["reasoning"]}")
    print(f"code grade : {result["Code Grade"]}")

testcase : {'task': 'Parse an AWS CloudFormation template and extract all S3 bucket logical IDs', 'format': 'python', 'solution_criteria': "Function should accept a CloudFormation JSON string and return a list of logical IDs for resources with type 'AWS::S3::Bucket'"}
output : 
import json
import sys

def extract_s3_buckets(template):
    if isinstance(template, str):
        template = json.loads(template)
    
    s3_buckets = []
    resources = template.get('Resources', {})
    
    for logical_id, resource in resources.items():
        if resource.get('Type') == 'AWS::S3::Bucket':
            s3_buckets.append(logical_id)
    
    return s3_buckets

if __name__ == '__main__':
    template_str = sys.stdin.read()
    buckets = extract_s3_buckets(template_str)
    print(json.dumps(buckets))

score : 7
reasoning : The solution correctly implements the core requirement of extracting S3 bucket logical IDs from a CloudFormation template. The logic is sound and handles the happy path well.